# Práctica guiada: del modelo legado al artefacto explícito

En la semana 3 usamos un payload **joblib** con tres claves: el estimador, los nombres de las features y la versión del modelo. En esta práctica vamos a separar responsabilidades:

- **model.joblib**: el objeto ejecutable;
- **manifest.json**: el contrato que permite saber si el objeto es compatible;
- código y pruebas: las reglas de validación y el comportamiento ante errores.

## Resultado esperado

Al terminar tendrás un manifiesto mínimo escrito como JSON, una lista de invariantes y una matriz de fallos que otra persona podrá implementar en la segunda clase.

## 1. Inspeccionar el punto de partida

Primero comprueba qué columnas recibe el módulo de inferencia de la semana 3. No cambies todavía el modelo ni ejecutes predicciones.

In [1]:
# para ver que el environment es el correcto
import sys
print(sys.executable)

c:\Users\marta\OpModelos_Master\MLV_V2_Correcto_icai-muiaap-operacion-de-modelos\semana4\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
import csv

#busca la ruta d eesta semanan
week_root = next( 
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "modules/04-model-packaging").is_dir()
)
# mira si la ruta del archi csv esta en semana4 (esta semanan) sino asume que esta en la semana3 
input_path = week_root / "assets/03-wine-quality/inference_samples.csv"
if not input_path.exists():
    input_path = week_root.parent / "semana3/assets/03-wine-quality/inference_samples.csv"

with input_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
print(f"Fichero: {input_path}")
print(f"Columnas: {reader.fieldnames}")
print(f"Filas de ejemplo: {len(rows)}")

Fichero: c:\Users\marta\OpModelos_Master\MLV_V2_Correcto_icai-muiaap-operacion-de-modelos\semana3\assets\03-wine-quality\inference_samples.csv
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Filas de ejemplo: 5


## 2. Diseñar la frontera del artefacto

Completa esta tabla antes de escribir código. La pregunta clave es: **¿qué necesita un proceso de inferencia para decidir que puede utilizar el artefacto?**

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | **model.joblib** | Evita tener que volver a entrenar el modelo  |
| Versión del esquema | manifest.json | tener una estructura d edatos incompatible   |
| Orden y nombres de features | manifest.json | introducir los datos en orden distinto al establecido, lo que puede llevar a errores en la salida si el tipo de dato era compatible pero no el significado/magintud de los datos |
| Versión del preprocesado | manifest.json | Evita aplicar transformaciones distintas de las utilizadas durante el entrenamiento |
| Etiquetas de salida permitidas |manifest.json | Evita devolver reultados que no se sepan interpretar |
| Versión del modelo | manifest.json | evita confundir modelos  |
| Tipo de estimador | manifest.json | Evita cargar o utilizar un objeto de un tipo inesperado |

## 3. Práctica 2 — Construir un manifiesto mínimo

Completa el borrador de Python de la siguiente celda y ejecútalo para producir un JSON legible. Mantén los nombres de campo estables: el manifiesto es una interfaz entre quien empaqueta el modelo y quien lo ejecuta.

~~~json
{
  "schema_version": "",
  "model_version": "",
  "preprocessing_version": "",
  "feature_names": [],
  "output_labels": [],
  "estimator_type": ""
}
~~~

Comprueba antes de continuar que:

- aparecen exactamente las seis claves obligatorias;
- `feature_names` tiene once nombres y no incluye `sample_id`;
- `output_labels` representa las categorías que la interfaz podrá devolver.

Decide también qué debe pasar si:

1. falta **manifest.json**;
2. aparecen features desconocidas o en un orden distinto;
3. el modelo devuelve una etiqueta que no está en **output_labels**;
4. una fila del CSV no cumple el contrato de entrada.

In [3]:
import json

feature_names = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]

# No modifiques este orden: procede del contrato de la semana 3.
legacy_model_version = "wine-quality-rf-demo-v1"

manifest_draft = {
    "schema_version": "wine-quality-bundle-v1", # ARTIFACT_SCHEMA_VERSION = "wine-quality-bundle-v1" <- artifact.py <- seman4 
    "model_version": legacy_model_version,
    "preprocessing_version": "wine-red-features-v1", # PREPROCESSING_VERSION = "wine-red-features-v1" <- preprocess.py <- semana3/4
    "feature_names": feature_names,
    "output_labels": ["low", "medium", "high"],
    "estimator_type": "RandomForestClassifier",
}

required_keys = {
    "schema_version",
    "model_version",
    "preprocessing_version",
    "feature_names",
    "output_labels",
    "estimator_type",
}

assert set(manifest_draft) == required_keys
assert len(manifest_draft["feature_names"]) == 11
assert "sample_id" not in manifest_draft["feature_names"]

print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))
print("\nLa estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.")

{
  "schema_version": "wine-quality-bundle-v1",
  "model_version": "wine-quality-rf-demo-v1",
  "preprocessing_version": "wine-red-features-v1",
  "feature_names": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
  ],
  "output_labels": [
    "low",
    "medium",
    "high"
  ],
  "estimator_type": "RandomForestClassifier"
}

La estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.


## 4. Práctica 3 — Anticipar fallos

Antes de ver la demo del profesorado, completa la tabla. No describas solo el error: indica en qué frontera debería detenerse el flujo.

| Cambio | Frontera que debe rechazarlo | Mensaje útil esperado | ¿Se escribe CSV? |
| --- | --- | --- | --- |
| Se intercambian `density` y `alcohol` | Carga del bundle. Validación del esquema de entrada: nombres y orden de las features | El orden de feature_names no es el esperado. Se esperaba features_names: ... | No |
| Cambia `preprocessing_version` | carga del bundle. Se valida el manifiesto | La version del manifiesto no es el esperado (wine-red-features-v1) | No |
| El modelo devuelve `unknown` | validacion de salida | Se devuleve unknown que no esta entre las etiquetas permitidas | No |
| La petición incluye `ph = 99` | validacion de entrada | El valor del pH no esta en los rangos permitidos | No |

## 5. Práctica 4 — Preparar la implementación de la segunda clase

Antes de pasar al taller, deja cerradas estas decisiones:

- ¿Qué campos son obligatorios y cuáles tienen valores por defecto? Los 6 campos del manifiesto son obligatorio y sin valores por defecto
- ¿Aceptarás campos desconocidos en el manifiesto? No
- ¿Validarás el manifiesto antes de cargar el objeto serializado? Si
- ¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas? Despues de validar que todos los parametros de salida son correcos
- ¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle? Un error en el que se indique donde ha fallado
- ¿Qué ocurre si una segunda fila es inválida después de haber procesado una primera fila correcta? No se realiza la escritura del csv y se manda un mensaje de error

La segunda clase convertirá estas decisiones en **ArtifactManifest**, **save_model_bundle**, **load_model_bundle**, inferencia y pruebas.